---
title: Fastp QC Summary
subtitle: The outcome of data QC
date: "9999-12-31"
edit_url: null
---


In [214]:
import json
import os
import sys
import time
from harpy.report.html import print_html, StatsBox


In [31]:
indir = "reports/data/fastp"

In [94]:
indir = "/home/pdimens/Projects/harpy/QC/reports/data/fastp"

In [9]:
print_html(fastp_summary(indir))

In [200]:
import os
import json
import polars as pl

def human_format(num):
        if num >= 1e9:
            return f"{num/1e9:.2f}G"
        elif num >= 1e6:
            return f"{num/1e6:.2f}M"
        elif num >= 1e3:
            return f"{num/1e3:.2f}K"
        else:
            return str(num)

class FastpResults():
    def __init__(self, report_dir):
        # Collect all JSON report files
        stats = {
            "Sample"         : [],
            "Reads (Before)" : [],
            "Reads (After)"  : [],
            "Bases (Before)" : [],
            "Bases (After)"  : [],
            "% Q20 (Before)" : [],
            "% Q20 (After)"  : [],
            "% Q30 (Before)" : [],
            "% Q30 (After)"  : [],
            "% GC (Before)"  : [],
            "% GC (After)"   : []
        }
        self.mean_qual_curves = []
        self.max_len_before = 0
        self.max_len_after = 0
        self.gc_curves = []
        self.max_len_gc_before = 0
        self.max_len_gc_after = 0
        self.mean_qual_curves_r2 = []
        self.max_len_before_r2 = 0
        self.max_len_after_r2 = 0
        self.gc_curves_r2 = []
        self.max_len_gc_before_r2 = 0
        self.max_len_gc_after_r2 = 0
        json_files = [f for f in os.listdir(report_dir) if f.endswith('.json')]
        for jf in json_files:
            path = os.path.join(report_dir, jf)
            with open(path) as f:
                data = json.load(f)
                summary = data.get('summary', {})
                before = summary.get('before_filtering', {})
                after = summary.get('after_filtering', {})
                # Extract quality and GC curves for read1
                qual_curve_before = data.get('read1_before_filtering', {}).get('quality_curves', {}).get('mean', [])
                qual_curve_after = data.get('read1_after_filtering', {}).get('quality_curves', {}).get('mean', [])
                gc_curve_before = data.get('read1_before_filtering', {}).get('content_curves', {}).get('GC', [])
                gc_curve_after = data.get('read1_after_filtering', {}).get('content_curves', {}).get('GC', [])
                self.mean_qual_curves.append({
                    'file': jf.replace('.fastp.json', ''),
                    'curve_before': qual_curve_before,
                    'curve_after': qual_curve_after
                })
                self.gc_curves.append({
                    'file': jf.replace('.fastp.json', ''),
                    'curve_before': gc_curve_before,
                    'curve_after': gc_curve_after
                })
                #if len(qual_curve_before) > max_len_before:
                #    max_len_before = len(qual_curve_before)
                self.max_len_before = max(self.max_len_before, len(qual_curve_before))
                self.max_len_after = max(len(qual_curve_after), self.max_len_after)
                self.max_len_gc_before = max(len(gc_curve_before), self.max_len_gc_before)
                self.max_len_gc_after = max(len(gc_curve_after), self.max_len_gc_after) 

                # Extract quality and GC curves for read2 if present
                qual_curve_before_r2 = data.get('read2_before_filtering', {}).get('quality_curves', {}).get('mean', [])
                qual_curve_after_r2 = data.get('read2_after_filtering', {}).get('quality_curves', {}).get('mean', [])
                gc_curve_before_r2 = data.get('read2_before_filtering', {}).get('content_curves', {}).get('GC', [])
                gc_curve_after_r2 = data.get('read2_after_filtering', {}).get('content_curves', {}).get('GC', [])
                if qual_curve_before_r2 or qual_curve_after_r2 or gc_curve_before_r2 or gc_curve_after_r2:
                    self.mean_qual_curves_r2.append({
                        'file': jf.replace('.fastp.json', ''),
                        'curve_before': qual_curve_before_r2,
                        'curve_after': qual_curve_after_r2
                    })
                    self.gc_curves_r2.append({
                        'file': jf.replace('.fastp.json', ''),
                        'curve_before': gc_curve_before_r2,
                        'curve_after': gc_curve_after_r2
                    })
                    self.max_len_before_r2 = max(len(qual_curve_before_r2), self.max_len_before_r2)
                    self.max_len_after_r2 = max(len(qual_curve_after_r2), self.max_len_after_r2)
                    self.max_len_gc_before_r2 = max(len(gc_curve_before_r2), self.max_len_gc_before_r2)
                    self.max_len_gc_after_r2 = max(len(gc_curve_after_r2), self.max_len_gc_after_r2)

                    stats["Sample"].append(jf.replace('.fastp.json', ''))
                    stats["Reads (Before)"].append(before.get('total_reads', 0))
                    stats["Reads (After)"].append(after.get('total_reads', 0))
                    stats["Bases (Before)"].append(before.get('total_bases', 0))
                    stats["Bases (After)"].append(after.get('total_bases', 0))
                    stats["% Q20 (Before)"].append(round(before.get('q20_rate', 0) * 100, 2))
                    stats["% Q20 (After)"].append(round(after.get('q20_rate', 0) * 100, 2))
                    stats["% Q30 (Before)"].append(round(before.get('q30_rate', 0) * 100, 2))
                    stats["% Q30 (After)"].append(round(after.get('q30_rate', 0) * 100, 2))
                    stats["% GC (Before)"].append(round(before.get('gc_content', 0) * 100, 2))
                    stats["% GC (After)"].append(round(after.get('gc_content', 0) * 100, 2))
        
        self.stats =  pl.DataFrame(stats)


In [ ]:
a = FastpResults(indir)
#TODO format curves into tables of samplebefore sampleafter
a.gc_curves_r2

[{'file': 'sample1',
  'curve_before': [0.663925,
   0.398379,
   0.402258,
   0.397046,
   0.228976,
   0.387403,
   0.404241,
   0.404371,
   0.627583,
   0.33005,
   0.493602,
   0.397631,
   0.489538,
   0.487393,
   0.442978,
   0.422141,
   0.406004,
   0.395256,
   0.413232,
   0.405934,
   0.418159,
   0.411319,
   0.415478,
   0.417445,
   0.409742,
   0.415678,
   0.416552,
   0.411122,
   0.41625,
   0.402619,
   0.416962,
   0.408334,
   0.411975,
   0.412124,
   0.408499,
   0.414574,
   0.414316,
   0.407744,
   0.418667,
   0.405297,
   0.414742,
   0.406983,
   0.411035,
   0.411202,
   0.408347,
   0.414009,
   0.41167,
   0.409988,
   0.416765,
   0.407235,
   0.416455,
   0.41119,
   0.412274,
   0.41084,
   0.407192,
   0.412699,
   0.412549,
   0.412937,
   0.413972,
   0.402934,
   0.412055,
   0.41161,
   0.410681,
   0.414002,
   0.405367,
   0.413845,
   0.410156,
   0.411146,
   0.417024,
   0.405448,
   0.412393,
   0.412002,
   0.409424,
   0.411353,
   0.40

In [ ]:
(#TODO UPDATE FOR NEW INFO
    StatsBox()
    .add(df['n_snp'].sum(), 'Total SNPs')
    .add(round(df['n_snp'].mean(), 0), 'Mean SNPs')
    .add(df['n_snp'].median(), 'Median SNPs')
    .add(df['block_length'].max(), 'Longest Haplotype')
    .add(nxx(df['block_length'], 50) / 1000, 'N50', units = "kb")
    .add(nxx(df['block_length'], 75) / 1000, 'N75', units = "kb")
    .add(nxx(df['block_length'], 90) / 1000, 'N90', units = "kb")
).render()

In [ ]:

    html = f'''
<div class="{root_class}">
<style>
    .{root_class} {{ font-family: 'Segoe UI', Arial, sans-serif; }}
    .{root_class} h1, .{root_class} h2 {{ color: #2c3e50; }}
    .{root_class} table {{ border-collapse: collapse; width: 100%; margin-bottom: 2em; background: #fff; }}
    .{root_class} th, .{root_class} td {{ border: 1px solid #e1e4e8; padding: 0.75em 1em; text-align: center; }}
    .{root_class} th {{ background: #f3f6fa; color: #34495e; }}
    .{root_class} tr:nth-child(even) {{ background: #f9fafb; }}
    .{root_class} a {{ color: #2980b9; text-decoration: none; }}
    .{root_class} a:hover {{ text-decoration: underline; }}
    .{root_class} .chart-container {{ width: 100%; max-width: none; aspect-ratio: 4/1; }}
    .{root_class} .row-charts-table {{ width: 100%; margin-bottom: 2em; background: none; border: none; }}
    .{root_class} .row-charts-table td {{ border: none; vertical-align: top; width: 50%; }}
</style>
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
<h2>FASTQ Aggregate Summary (''' + fastp_version + f''')</h2>
    <table>
        <thead>
            <tr>
                <th>File</th>
                <th>Total Reads (Before)</th>
                <th>Total Reads (After)</th>
                <th>Total Bases (Before)</th>
                <th>Total Bases (After)</th>
                <th>Q20 Rate (Before)</th>
                <th>Q20 Rate (After)</th>
                <th>Q30 Rate (Before)</th>
                <th>Q30 Rate (After)</th>
                <th>GC Content (Before)</th>
                <th>GC Content (After)</th>
                <th>HTML Report</th>
            </tr>
        </thead>
        <tbody>
'''

    def human_format(num):
        if num >= 1e9:
            return f"{num/1e9:.2f}G"
        elif num >= 1e6:
            return f"{num/1e6:.2f}M"
        elif num >= 1e3:
            return f"{num/1e3:.2f}K"
        else:
            return str(num)

    for s in stats:
        html += f'<tr>'
        html += f'<td>{s["file"]}</td>'
        html += f'<td>{human_format(s["total_reads_before"])}</td>'
        html += f'<td>{human_format(s["total_reads_after"])}</td>'
        html += f'<td>{human_format(s["total_bases_before"])}</td>'
        html += f'<td>{human_format(s["total_bases_after"])}</td>'
        html += f'<td>{s["q20_rate_before"]:.2f}%</td>'
        html += f'<td>{s["q20_rate_after"]:.2f}%</td>'
        html += f'<td>{s["q30_rate_before"]:.2f}%</td>'
        html += f'<td>{s["q30_rate_after"]:.2f}%</td>'
        html += f'<td>{s["gc_content_before"]:.2f}%</td>'
        html += f'<td>{s["gc_content_after"]:.2f}%</td>'
        html += f'<td><a href="{s["html_report"]}">View</a></td>'
        html += '</tr>'

    html += f'''
        </tbody>
    </table>
    <table class="row-charts-table">
        <tr>
            <td><div id="{eid('meanQualPlotBefore')}" style="width:100%;height:400px;"></div></td>
            <td><div id="{eid('meanQualPlotAfter')}" style="width:100%;height:400px;"></div></td>
        </tr>
'''
    if mean_qual_curves_r2:
        html += f'        <tr>\n'
        html += f'            <td><div id="{eid("meanQualPlotBeforeR2")}" style="width:100%;height:400px;"></div></td>\n'
        html += f'            <td><div id="{eid("meanQualPlotAfterR2")}" style="width:100%;height:400px;"></div></td>\n'
        html += '        </tr>\n'
    html += f'        <tr>\n'
    html += f'            <td><div id="{eid("gcCurvePlotBefore")}" style="width:100%;height:400px;"></div></td>\n'
    html += f'            <td><div id="{eid("gcCurvePlotAfter")}" style="width:100%;height:400px;"></div></td>\n'
    html += '        </tr>\n'
    if gc_curves_r2:
        html += f'        <tr>\n'
        html += f'            <td><div id="{eid("gcCurvePlotBeforeR2")}" style="width:100%;height:400px;"></div></td>\n'
        html += f'            <td><div id="{eid("gcCurvePlotAfterR2")}" style="width:100%;height:400px;"></div></td>\n'
        html += '        </tr>\n'
    html += '    </table>\n'

    html += f'''
    <div class="chart-container" style="width:100%; max-width:none; aspect-ratio: 4/1;">
        <canvas id="{eid('qRateChart')}" style="height:200px;"></canvas>
    </div>
    <script>
    (function() {{
        const files = ''' + json.dumps([s['file'] for s in stats]) + ''';
        const totalReadsBefore = ''' + json.dumps([s['total_reads_before'] for s in stats]) + ''';
        const totalReadsAfter = ''' + json.dumps([s['total_reads_after'] for s in stats]) + ''';
        const totalBasesBefore = ''' + json.dumps([s['total_bases_before'] for s in stats]) + ''';
        const totalBasesAfter = ''' + json.dumps([s['total_bases_after'] for s in stats]) + ''';
        const q20Before = ''' + json.dumps([s['q20_rate_before'] for s in stats]) + ''';
        const q20After = ''' + json.dumps([s['q20_rate_after'] for s in stats]) + ''';
        const q30Before = ''' + json.dumps([s['q30_rate_before'] for s in stats]) + ''';
        const q30After = ''' + json.dumps([s['q30_rate_after'] for s in stats]) + ''';
        // Plotly mean quality curves (before)
        const meanQualCurves = ''' + json.dumps(mean_qual_curves) + ''';
        const maxLenBefore = ''' + str(max_len_before) + ''';
        const maxLenAfter = ''' + str(max_len_after) + f''';
        const qualLabelsBefore = Array.from({{length: maxLenBefore}}, (_, i) => i + 1);
        const qualLabelsAfter = Array.from({{length: maxLenAfter}}, (_, i) => i + 1);
        const plotlyTracesBefore = meanQualCurves.map((item, idx) => {{
            const before = item.curve_before || [];
            const beforePad = before.concat(Array(maxLenBefore - before.length).fill(null));
            return {{
                x: qualLabelsBefore,
                y: beforePad,
                mode: 'lines',
                name: item.file,
                line: {{ width: 1 }}
            }};
        }});
        Plotly.newPlot('{eid("meanQualPlotBefore")}', plotlyTracesBefore, {{
            title: 'Mean Quality Curve (Read1, Before Filtering)',
            xaxis: {{ title: '' }},
            yaxis: {{ title: 'Mean Quality', rangemode: 'tozero' }},
            legend: {{ orientation: 'h' }},
            margin: {{ t: 50, l: 60, r: 30, b: 60 }}
        }}, {{responsive: true}});
        // Plotly mean quality curves (after)
        const plotlyTracesAfter = meanQualCurves.map((item, idx) => {{
            const after = item.curve_after || [];
            const afterPad = after.concat(Array(maxLenAfter - after.length).fill(null));
            return {{
                x: qualLabelsAfter,
                y: afterPad,
                mode: 'lines',
                name: item.file,
                line: {{ width: 1 }}
            }};
        }});
        Plotly.newPlot('{eid("meanQualPlotAfter")}', plotlyTracesAfter, {{
            title: 'Mean Quality Curve (Read1, After Filtering)',
            xaxis: {{ title: '' }},
            yaxis: {{ title: 'Mean Quality', rangemode: 'tozero' }},
            legend: {{ orientation: 'h' }},
            margin: {{ t: 50, l: 60, r: 30, b: 60 }}
        }}, {{responsive: true}});
        // Plotly GC content curves (before)
        const gcCurves = ''' + json.dumps(gc_curves) + ''';
        const maxLenGCBefore = ''' + str(max_len_gc_before) + ''';
        const maxLenGCAfter = ''' + str(max_len_gc_after) + f''';
        const gcLabelsBefore = Array.from({{length: maxLenGCBefore}}, (_, i) => i + 1);
        const gcLabelsAfter = Array.from({{length: maxLenGCAfter}}, (_, i) => i + 1);
        const plotlyTracesGCBefore = gcCurves.map((item, idx) => {{
            const before = item.curve_before || [];
            const beforePad = before.concat(Array(maxLenGCBefore - before.length).fill(null));
            return {{
                x: gcLabelsBefore,
                y: beforePad,
                mode: 'lines',
                name: item.file,
                line: {{ width: 1 }}
            }};
        }});
        Plotly.newPlot('{eid("gcCurvePlotBefore")}', plotlyTracesGCBefore, {{
            title: 'GC Content Curve (Read1, Before Filtering)',
            xaxis: {{ title: '' }},
            yaxis: {{ title: 'GC %', rangemode: 'tozero' }},
            legend: {{ orientation: 'h' }},
            margin: {{ t: 50, l: 60, r: 30, b: 60 }}
        }}, {{responsive: true}});
        // Plotly GC content curves (after)
        const plotlyTracesGCAfter = gcCurves.map((item, idx) => {{
            const after = item.curve_after || [];
            const afterPad = after.concat(Array(maxLenGCAfter - after.length).fill(null));
            return {{
                x: gcLabelsAfter,
                y: afterPad,
                mode: 'lines',
                name: item.file,
                line: {{ width: 1 }}
            }};
        }});
        Plotly.newPlot('{eid("gcCurvePlotAfter")}', plotlyTracesGCAfter, {{
            title: 'GC Content Curve (Read1, After Filtering)',
            xaxis: {{ title: '' }},
            yaxis: {{ title: 'GC %', rangemode: 'tozero' }},
            legend: {{ orientation: 'h' }},
            margin: {{ t: 50, l: 60, r: 30, b: 60 }}
        }}, {{responsive: true}});
        // Plotly mean quality and GC curves for read2 (if any)
        const meanQualCurvesR2 = ''' + json.dumps(mean_qual_curves_r2) + ''';
        const gcCurvesR2 = ''' + json.dumps(gc_curves_r2) + ''';
        const maxLenBeforeR2 = ''' + str(max_len_before_r2) + ''';
        const maxLenAfterR2 = ''' + str(max_len_after_r2) + ''';
        const maxLenGCBeforeR2 = ''' + str(max_len_gc_before_r2) + ''';
        const maxLenGCAfterR2 = ''' + str(max_len_gc_after_r2) + f'''; 
        if (meanQualCurvesR2.length > 0 || gcCurvesR2.length > 0) {{
            const qualLabelsBeforeR2 = Array.from({{length: maxLenBeforeR2}}, (_, i) => i + 1);
            const qualLabelsAfterR2 = Array.from({{length: maxLenAfterR2}}, (_, i) => i + 1);
            const gcLabelsBeforeR2 = Array.from({{length: maxLenGCBeforeR2}}, (_, i) => i + 1);
            const gcLabelsAfterR2 = Array.from({{length: maxLenGCAfterR2}}, (_, i) => i + 1);
            const plotlyTracesBeforeR2 = meanQualCurvesR2.map((item, idx) => {{
                const before = item.curve_before || [];
                const beforePad = before.concat(Array(maxLenBeforeR2 - before.length).fill(null));
                return {{
                    x: qualLabelsBeforeR2,
                    y: beforePad,
                    mode: 'lines',
                    name: item.file,
                    line: {{ width: 1 }}
                }};
            }});
            Plotly.newPlot('{eid("meanQualPlotBeforeR2")}', plotlyTracesBeforeR2, {{
                title: 'Mean Quality Curve (Read2, Before Filtering)',
                xaxis: {{ title: '' }},
                yaxis: {{ title: 'Mean Quality', rangemode: 'tozero' }},
                legend: {{ orientation: 'h' }},
                margin: {{ t: 50, l: 60, r: 30, b: 60 }}
            }}, {{responsive: true}});
            const plotlyTracesAfterR2 = meanQualCurvesR2.map((item, idx) => {{
                const after = item.curve_after || [];
                const afterPad = after.concat(Array(maxLenAfterR2 - after.length).fill(null));
                return {{
                    x: qualLabelsAfterR2,
                    y: afterPad,
                    mode: 'lines',
                    name: item.file,
                    line: {{ width: 1 }}
                }};
            }});
            Plotly.newPlot('{eid("meanQualPlotAfterR2")}', plotlyTracesAfterR2, {{
                title: 'Mean Quality Curve (Read2, After Filtering)',
                xaxis: {{ title: '' }},
                yaxis: {{ title: 'Mean Quality', rangemode: 'tozero' }},
                legend: {{ orientation: 'h' }},
                margin: {{ t: 50, l: 60, r: 30, b: 60 }}
            }}, {{responsive: true}});
            // GC content curves for read2
            const plotlyTracesGCBeforeR2 = gcCurvesR2.map((item, idx) => {{
                const before = item.curve_before || [];
                const beforePad = before.concat(Array(maxLenGCBeforeR2 - before.length).fill(null));
                return {{
                    x: gcLabelsBeforeR2,
                    y: beforePad,
                    mode: 'lines',
                    name: item.file,
                    line: {{ width: 1 }}
                }};
            }});
            Plotly.newPlot('{eid("gcCurvePlotBeforeR2")}', plotlyTracesGCBeforeR2, {{
                title: 'GC Content Curve (Read2, Before Filtering)',
                xaxis: {{ title: '' }},
                yaxis: {{ title: 'GC %', rangemode: 'tozero' }},
                legend: {{ orientation: 'h' }},
                margin: {{ t: 50, l: 60, r: 30, b: 60 }}
            }}, {{responsive: true}});
            const plotlyTracesGCAfterR2 = gcCurvesR2.map((item, idx) => {{
                const after = item.curve_after || [];
                const afterPad = after.concat(Array(maxLenGCAfterR2 - after.length).fill(null));
                return {{
                    x: gcLabelsAfterR2,
                    y: afterPad,
                    mode: 'lines',
                    name: item.file,
                    line: {{ width: 1 }}
                }};
            }});
            Plotly.newPlot('{eid("gcCurvePlotAfterR2")}', plotlyTracesGCAfterR2, {{
                title: 'GC Content Curve (Read2, After Filtering)',
                xaxis: {{ title: '' }},
                yaxis: {{ title: 'GC %', rangemode: 'tozero' }},
                legend: {{ orientation: 'h' }},
                margin: {{ t: 50, l: 60, r: 30, b: 60 }}
            }}, {{responsive: true}});
        }}
        // Q20/Q30 chart (grouped bar)
        new Chart(document.getElementById('{eid("qRateChart")}'), {{
            type: 'bar',
            data: {{
                labels: files,
                datasets: [
                    {{ label: 'Q20 Rate (Before)', data: q20Before, backgroundColor: '#fab1a0' }},
                    {{ label: 'Q20 Rate (After)', data: q20After, backgroundColor: '#e17055' }},
                    {{ label: 'Q30 Rate (Before)', data: q30Before, backgroundColor: '#b2bec3' }},
                    {{ label: 'Q30 Rate (After)', data: q30After, backgroundColor: '#2ecc71' }}
                ]
            }},
            options: {{
                responsive: true,
                maintainAspectRatio: false,
                plugins: {{ legend: {{ position: 'top' }} }},
                scales: {{ y: {{ beginAtZero: true, max: 100 }} }},
                animation: false
            }}
        }});
    }})();
    </script>
</div>
'''
    return html